# Figure 1 - Panel G: disease / pathway enrichment ridgeplot (v2)

**Reselected** (point 3d): the full enrichment survey shows the "disease" ontologies have no
neurodevelopment-disorder signal (HPO 0 sig; DO only off-target myopathy; **DisGeNET dominated
by off-target cancers** - the oligodendroglioma terms are glioma and were the redundant duplicate).
The strong, coherent, story-relevant disease-biology terms are kept: Reactome **cellular stress / WNT / senescence /
RHO-GTPase signalling** (cellular responses to stress is the single strongest enrichment, padj 7.9e-6;
RHO GTPases activate PKNs ties APA to neuronal cytoskeletal/morphogenesis signalling and contains
Cdc42/RAC1), KEGG **neurodegeneration**, Hallmark **Myc targets**. Prion disease was dropped from the
main panel (still significant, but less narrative-relevant); DisGeNET / oligodendroglioma is moved to
the supplementary full-enrichment panel.

Other changes (points 2, 3): density now uses **all** genes in each set (not core-enrichment);
**one shared** ORA-padj colour scale across all terms; faceted by database so **Hallmark gets room**.

- Backs draft **Section 1.3 (Figure 1G)**. Kernel: `r_downstream`.
- Input: `ORA/{gsea,ora}_{reactome,kegg,hallmark}_neglog10pval_chisq_optimized2.rds`.
- Output: `.../Fig1/fig1g_disease_enrichment/`.


## Locations

Paths are relative to the data directory given by the environment variable `M26_DATA_ROOT` (see the README); figure panels are written to `M26_RESULTS_ROOT`.

In [ ]:
# Locations (see README): M26_DATA_ROOT is required; M26_RESULTS_ROOT defaults to <repo>/results
REPO_ROOT <- Sys.getenv("M26_REPO_ROOT")
if (!nzchar(REPO_ROOT)) {
    REPO_ROOT <- normalizePath(getwd())
    while (!file.exists(file.path(REPO_ROOT, "environment", "downstr.yml")) && REPO_ROOT != dirname(REPO_ROOT)) {
        REPO_ROOT <- dirname(REPO_ROOT)
    }
    Sys.setenv(M26_REPO_ROOT = REPO_ROOT)
}
DATA_ROOT <- Sys.getenv("M26_DATA_ROOT")
if (!nzchar(DATA_ROOT)) stop("Set M26_DATA_ROOT (see README)")
RESULTS_ROOT <- Sys.getenv("M26_RESULTS_ROOT", file.path(REPO_ROOT, "results"))

In [ ]:
source(paste0(REPO_ROOT, "/figures/style/paper_fig_style.R"))
suppressPackageStartupMessages({
    library(clusterProfiler)
    library(dplyr)
    library(stringr)
    library(ggridges)
    library(scales)
})
ORA_DIR <- paste0(DATA_ROOT, "/3utr/wildtypes_analysis/2026_analyses/utrcalling/DirMult_optimized/ORA")
PANEL <- "fig1g_disease_enrichment"
FIG_W <- HALF_W_IN
FIG_H <- mm_to_in(58) # 85 x 58 mm (half-grid)

build_long <- function(gsea_obj, ora_obj, descs) {
    gl <- gsea_obj@geneList
    gs <- gsea_obj@geneSets
    gres <- gsea_obj@result
    ores <- ora_obj@result
    out <- list()
    for (d in descs) {
        gid <- gres$ID[match(d, gres$Description)]
        if (is.na(gid)) {
            gid <- ores$ID[match(d, ores$Description)]
        }
        if (is.na(gid) || is.null(gs[[gid]])) {
            message("skip: ", d)
            next
        }
        genes <- intersect(gs[[gid]], names(gl))
        padj <- ores$p.adjust[match(d, ores$Description)]
        out[[d]] <- data.frame(
            term = d,
            neglog10p = as.numeric(gl[genes]),
            ora_padj = padj,
            n = length(genes),
            stringsAsFactors = FALSE
        )
    }
    do.call(rbind, out)
}

ld <- function(tag, kind) {
    readRDS(file.path(
        ORA_DIR,
        sprintf("%s_%s_neglog10pval_chisq_optimized2.rds", kind, tag)
    ))
}
sources <- list(
    list(
        db = "Reactome",
        gsea = ld("reactome", "gsea"),
        ora = ld("reactome", "ora"),
        descs = c(
            "Cellular responses to stress",
            "Signaling by WNT",
            "DNA Damage/Telomere Stress Induced Senescence",
            "RHO GTPases activate PKNs"
        )
    ),
    list(
        db = "KEGG",
        gsea = ld("kegg", "gsea"),
        ora = ld("kegg", "ora"),
        descs = c("Pathways of neurodegeneration - multiple diseases")
    ),
    list(
        db = "Hallmark",
        gsea = ld("hallmark", "gsea"),
        ora = ld("hallmark", "ora"),
        descs = c("Myc Targets V1")
    )
)

In [ ]:
df <- do.call(
    rbind,
    lapply(sources, function(s) {
        d <- build_long(s$gsea, s$ora, s$descs)
        if (is.null(d)) {
            return(NULL)
        }
        d$database <- s$db
        d
    })
)
# order: database (Reactome, KEGG, Hallmark), then ORA padj within db
ord <- df %>%
    distinct(term, database, ora_padj) %>%
    arrange(
        factor(database, levels = c("Reactome", "KEGG", "Hallmark")),
        ora_padj
    )
ord$label <- sprintf(
    "%s  (n=%d)",
    str_wrap(ord$term, 26),
    sapply(ord$term, function(t) df$n[match(t, df$term)])
)
df$label <- factor(ord$label[match(df$term, ord$term)], levels = rev(ord$label))
df$database <- factor(df$database, levels = c("Reactome", "KEGG", "Hallmark"))
cat("terms x genes:\n")
print(ord[, c("database", "term", "ora_padj")])

In [ ]:
brks <- signif(
    exp(seq(log(min(df$ora_padj)), log(max(df$ora_padj)), length.out = 4)),
    1
)
p <- ggplot(df, aes(x = neglog10p, y = label, fill = ora_padj)) +
    geom_density_ridges(
        scale = 1.3,
        alpha = 0.92,
        color = "grey35",
        linewidth = 0.25,
        rel_min_height = 0.008,
        bandwidth = 0.5
    ) +
    facet_grid(
        database ~ .,
        scales = "free_y",
        space = "free_y",
        switch = "y"
    ) +
    scale_fill_gradientn(
        colours = rev(c("#ffffcc", "#fed976", "#fd8d3c", "#e31a1c", "#800026")),
        trans = "log10",
        limits = range(df$ora_padj),
        breaks = brks,
        labels = formatC(brks, format = "g", digits = 1),
        name = expression(ORA * " " * italic(p)[adj]),
        guide = guide_colourbar(
            barwidth = 0.45,
            barheight = 3.5,
            ticks.colour = "grey30"
        )
    ) +
    geom_vline(
        xintercept = -log10(0.05),
        linetype = "dashed",
        color = "black",
        linewidth = 0.4
    ) +
    labs(
        x = expression(
            "-" * "log"[10] * " " * italic(p) * "-value  (per gene in set)"
        ),
        y = NULL
    ) +
    coord_cartesian(xlim = c(0, 12)) +
    theme_paper +
    theme(
        axis.text.y = element_text(size = FONT_LEGEND, lineheight = 0.85),
        legend.position = "right",
        panel.grid.major.x = element_line(linewidth = 0.25, color = "gray90"),
        panel.grid.major.y = element_blank(),
        strip.text.y.left = element_text(
            size = FONT_TICK,
            face = "bold",
            angle = 90
        ),
        strip.background = element_rect(fill = "#F0F0F0", linewidth = 0.4),
        strip.placement = "outside",
        panel.spacing.y = unit(2, "pt")
    )

save_and_show(p, PANEL, FIG_W, FIG_H)

## Variant: chisq∩permutation intersection foreground

Same faceted disease/pathway ridgeplot recoloured from the **chisq ∩ permutation** intersection ORA
(mouse). All curated terms remain significant under the stricter foreground (Reactome: cellular
responses to stress 1e-3, WNT 3e-3, DNA-damage senescence 3e-3, RHO PKNs 9e-3; KEGG neurodegeneration
0.027; Hallmark Myc 1e-6). Saved `fig1g_disease_enrichment__chisqperm`.

In [ ]:
# --- fig1g on the chisq∩perm intersection foreground (gene-set fills from intersection ORA) ---
INT_DIR <- paste0(DATA_ROOT, "/3utr/wildtypes_analysis/2026_analyses/utrcalling/DirMult_optimized/ORA/chisq_perm_intersection")
ld_int <- function(db) readRDS(file.path(INT_DIR, sprintf("ora_%s_neglog10pval_chisqperm.rds", db)))
sources_int <- list(
    list(db = "Reactome", gsea = ld("reactome", "gsea"), ora = ld_int("reactome"),
         descs = c("Cellular responses to stress", "Signaling by WNT",
                   "DNA Damage/Telomere Stress Induced Senescence", "RHO GTPases activate PKNs")),
    list(db = "KEGG", gsea = ld("kegg", "gsea"), ora = ld_int("kegg"),
         descs = c("Pathways of neurodegeneration - multiple diseases")),
    list(db = "Hallmark", gsea = ld("hallmark", "gsea"), ora = ld_int("hallmark"),
         descs = c("Myc Targets V1")))
df_i <- do.call(rbind, lapply(sources_int, function(s) { d <- build_long(s$gsea, s$ora, s$descs)
    if (is.null(d)) return(NULL); d$database <- s$db; d }))
ord_i <- df_i %>% distinct(term, database, ora_padj) %>%
    arrange(factor(database, levels = c("Reactome","KEGG","Hallmark")), ora_padj)
ord_i$label <- sprintf("%s  (n=%d)", str_wrap(ord_i$term, 26), sapply(ord_i$term, function(t) df_i$n[match(t, df_i$term)]))
df_i$label <- factor(ord_i$label[match(df_i$term, ord_i$term)], levels = rev(ord_i$label))
df_i$database <- factor(df_i$database, levels = c("Reactome","KEGG","Hallmark"))
brks_i <- signif(exp(seq(log(min(df_i$ora_padj)), log(max(df_i$ora_padj)), length.out = 4)), 1)
p_i <- ggplot(df_i, aes(x = neglog10p, y = label, fill = ora_padj)) +
    geom_density_ridges(scale = 1.3, alpha = 0.92, color = "grey35", linewidth = 0.25, rel_min_height = 0.008, bandwidth = 0.5) +
    facet_grid(database ~ ., scales = "free_y", space = "free_y", switch = "y") +
    scale_fill_gradientn(colours = rev(c("#ffffcc","#fed976","#fd8d3c","#e31a1c","#800026")), trans = "log10",
        limits = range(df_i$ora_padj), breaks = brks_i, labels = formatC(brks_i, format = "g", digits = 1),
        name = expression(ORA*" "*italic(p)[adj]), guide = guide_colourbar(barwidth = 0.45, barheight = 3.5, ticks.colour = "grey30")) +
    geom_vline(xintercept = -log10(0.05), linetype = "dashed", color = "black", linewidth = 0.4) +
    labs(x = expression("-"*"log"[10]*" "*italic(p)*"-value  (per gene in set)"), y = NULL) +
    scale_x_continuous(expand = expansion(mult = c(0, 0.02)), breaks = seq(0, 12.5, 2.5)) +
    coord_cartesian(xlim = c(0, 12.5), clip = "on") + theme_paper +
    theme(axis.text.y = element_text(size = FONT_LEGEND, lineheight = 0.85), legend.position = "right",
        panel.border = element_rect(linewidth = 0.3, fill = NA),
        panel.grid.major.x = element_line(linewidth = 0.25, color = "gray90"), panel.grid.major.y = element_blank(),
        strip.text.y.left = element_text(size = FONT_TICK, face = "bold", angle = 90),
        strip.background = element_rect(fill = "#F0F0F0", linewidth = 0.4), strip.placement = "outside",
        panel.spacing.y = unit(2, "pt"))
save_and_show(p_i, "fig1g_disease_enrichment__chisqperm", FIG_W, FIG_H)
